# From Prompt to Persistence: Multi-Tenant Memory Schema

Companion notebook for **From Prompt to Persistence: Designing Multi-Tenant Agent Memory Schemas for SaaS**. Every SQL query from the article runs here against a real Oracle AI Database instance, in the order the article presents them: schema DDL, RLS policy, session context, sample data, canonical retrieval predicate, supersession, right-to-forget cascade, and the unified retrieval query.

## Prerequisites

- Oracle AI Database Free running at `localhost:1521/FREEPDB1` (or set `DB_DSN`)
- A user with `DB_DEVELOPER_ROLE`, `CREATE SESSION`, `CREATE TABLE`, and `EXECUTE ON DBMS_RLS` granted
- `EXECUTE ON CTX_DDL` if you want to extend with Oracle Text indexes
- ONNX embedding model loaded (`ALL_MINILM_L12_V2`, 384-dim); see the project README for the loader script
- A `.env` file with `DB_USER`, `DB_PASSWORD`, `DB_DSN`

The article's DDL declares vectors as `VECTOR(1024, FLOAT32)` as an illustrative dimension. This notebook uses `VECTOR(384, FLOAT32)` to match `ALL_MINILM_L12_V2`. The shape of the SQL is identical; only the dimension number changes.

## 1. Environment & connection

In [1]:
import os, json, hashlib, uuid
from datetime import datetime, timezone

import oracledb
from dotenv import load_dotenv

load_dotenv()

DB_USER     = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_DSN      = os.getenv("DB_DSN", "localhost:1521/FREEPDB1")

conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
print("Oracle version:", conn.version)
cur = conn.cursor()
# Pin the session timezone to UTC. The schema's valid_from/valid_until/
# created_at are plain TIMESTAMP columns written from SYSTIMESTAMP (DB tz = UTC);
# without this, a session in a different tz makes 'valid_until > SYSTIMESTAMP'
# misjudge superseded rows as still active.
cur.execute("ALTER SESSION SET TIME_ZONE = 'UTC'")

Oracle version: 23.26.1.0.0


## 2. The eight LTM tables (plus audit)

Eight typed tables sharing the same scope columns (`tenant_id`, `user_id`, `agent_id`, `thread_id`), the same lifecycle columns (`valid_from`, `valid_until`, `created_at`, `deleted_at`), and a `deletion_events` audit table for the right-to-forget cascade. We drop existing copies first so the cell is idempotent.

In [2]:
# Idempotent teardown.
TABLES = [
    "knowledge_base_chunk", "knowledge_base_document",
    "toolbox_memory", "workflow_memory", "summarization_memory",
    "conversation_memory", "entity_memory", "persona_memory",
    "guideline_memory", "deletion_events",
]
for t in TABLES:
    try:
        cur.execute(f"DROP TABLE {t} CASCADE CONSTRAINTS PURGE")
    except oracledb.DatabaseError:
        pass
print("Tables dropped (if they existed).")

Tables dropped (if they existed).


### 2a. `guideline_memory` (Procedural)

In [3]:
cur.execute("""
CREATE TABLE guideline_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64),              -- session identifier
  guideline_key   VARCHAR2(128) NOT NULL,
  guideline_value JSON          NOT NULL,
  version         NUMBER(10)    NOT NULL,
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64),
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

cur.execute("""
CREATE UNIQUE INDEX idx_guideline_active
  ON guideline_memory (
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN tenant_id      END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN user_id        END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN agent_id       END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN thread_id      END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN guideline_key END
  )
""")
print("guideline_memory created.")

guideline_memory created.


### 2b. `persona_memory` (Semantic)

In [4]:
cur.execute("""
CREATE TABLE persona_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64),
  persona_key     VARCHAR2(64)  NOT NULL,
  persona_value   JSON,                      -- nullable: erased on right-to-forget
  confidence      NUMBER(3,2),
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

cur.execute("""
CREATE UNIQUE INDEX idx_persona_active
  ON persona_memory (
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN tenant_id      END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN user_id        END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN agent_id       END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN thread_id      END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN persona_key END
  )
""")
print("persona_memory created.")

persona_memory created.


### 2c. `entity_memory` (Semantic)

In [5]:
cur.execute("""
CREATE TABLE entity_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64),
  subject         VARCHAR2(256) NOT NULL,
  predicate       VARCHAR2(64)  NOT NULL,
  content         CLOB,                      -- nullable: erased on right-to-forget
  content_hash    VARCHAR2(64)  NOT NULL,
  content_json    JSON,
  embedding       VECTOR(384, FLOAT32),
  confidence      NUMBER(3,2)   NOT NULL,
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64)  NOT NULL,
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

for ix in [
    "CREATE INDEX idx_entity_scope_active ON entity_memory (tenant_id, user_id, agent_id, thread_id, deleted_at, valid_until)",
    "CREATE INDEX idx_entity_dedup        ON entity_memory (tenant_id, user_id, agent_id, content_hash)",
    "CREATE INDEX idx_entity_subject      ON entity_memory (tenant_id, subject)",
    "CREATE INDEX idx_entity_content_json ON entity_memory (JSON_VALUE(content_json, '$.entity_type'))",
]:
    cur.execute(ix)

cur.execute("""
CREATE VECTOR INDEX idx_entity_embedding
  ON entity_memory (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH
  DISTANCE COSINE
  WITH TARGET ACCURACY 95
""")

# Oracle Text index for the lexical half of hybrid retrieval on entities.
# SYNC (ON COMMIT) keeps it current as rows are inserted/committed.
cur.execute("""
CREATE INDEX idx_entity_text ON entity_memory (content)
  INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')
""")
print("entity_memory created with 6 indexes (incl. Oracle Text on content).")

entity_memory created with 6 indexes (incl. Oracle Text on content).


### 2d. `conversation_memory` (Episodic, interval-partitioned)

In [6]:
cur.execute("""
CREATE TABLE conversation_memory (
  event_id        VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64)  NOT NULL,
  run_id          VARCHAR2(64)  NOT NULL,
  turn_index      NUMBER(10)    NOT NULL,
  event_type      VARCHAR2(32)  NOT NULL,
  role            VARCHAR2(32),
  payload         JSON          NOT NULL,
  token_cost      NUMBER(10),
  latency_ms      NUMBER(10),
  retention_class VARCHAR2(16)  DEFAULT 'short' NOT NULL,
  created_at      TIMESTAMP     NOT NULL
) PARTITION BY RANGE (created_at)
  INTERVAL (NUMTODSINTERVAL(1, 'DAY'))
  ( PARTITION p0 VALUES LESS THAN (TIMESTAMP '2026-01-01 00:00:00') )
""")

for ix in [
    "CREATE INDEX idx_conv_thread_turn ON conversation_memory (tenant_id, thread_id, turn_index)",
    "CREATE INDEX idx_conv_run         ON conversation_memory (tenant_id, run_id, turn_index)",
    "CREATE INDEX idx_conv_user_time   ON conversation_memory (tenant_id, user_id, created_at)",
]:
    cur.execute(ix)
print("conversation_memory created with day-interval partitioning.")

conversation_memory created with day-interval partitioning.


### 2e. `summarization_memory` (Episodic)

In [7]:
cur.execute("""
CREATE TABLE summarization_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  thread_id       VARCHAR2(64),
  task_type       VARCHAR2(64)  NOT NULL,
  title           VARCHAR2(256) NOT NULL,
  summary         CLOB,                      -- nullable: erased on right-to-forget
  key_steps       JSON,
  outcome         VARCHAR2(64),
  artifacts       JSON,
  embedding       VECTOR(384, FLOAT32),
  confidence      NUMBER(3,2)   NOT NULL,
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64)  NOT NULL,
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  completed_at    TIMESTAMP     NOT NULL,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

cur.execute("""
CREATE INDEX idx_summ_scope_task
  ON summarization_memory (tenant_id, user_id, agent_id, task_type, completed_at)
""")

cur.execute("""
CREATE VECTOR INDEX idx_summ_embedding
  ON summarization_memory (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH
  DISTANCE COSINE
  WITH TARGET ACCURACY 95
""")

# Oracle Text index for the lexical half of hybrid retrieval on summaries.
cur.execute("""
CREATE INDEX idx_summ_text ON summarization_memory (summary)
  INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')
""")
print("summarization_memory created (incl. Oracle Text on summary).")

summarization_memory created (incl. Oracle Text on summary).


### 2f. `workflow_memory` (Procedural)

In [8]:
cur.execute("""
CREATE TABLE workflow_memory (
  id               VARCHAR2(64)  PRIMARY KEY,
  tenant_id        VARCHAR2(64)  NOT NULL,
  user_id          VARCHAR2(64),
  agent_id         VARCHAR2(64),
  workflow_name    VARCHAR2(128) NOT NULL,
  description      CLOB,
  steps            JSON          NOT NULL,
  preconditions    JSON,
  expected_outcome JSON,
  success_count    NUMBER(10)    DEFAULT 0 NOT NULL,
  failure_count    NUMBER(10)    DEFAULT 0 NOT NULL,
  embedding        VECTOR(384, FLOAT32),
  confidence       NUMBER(3,2)   NOT NULL,
  written_by       VARCHAR2(64)  NOT NULL,
  source_event_id  VARCHAR2(64),
  version          NUMBER(10)    NOT NULL,
  superseded_by    VARCHAR2(64),
  valid_from       TIMESTAMP     NOT NULL,
  valid_until      TIMESTAMP,
  created_at       TIMESTAMP     NOT NULL,
  deleted_at       TIMESTAMP
)
""")

cur.execute("""
CREATE INDEX idx_workflow_scope_name
  ON workflow_memory (tenant_id, user_id, agent_id, workflow_name)
""")

cur.execute("""
CREATE VECTOR INDEX idx_workflow_embedding
  ON workflow_memory (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH
  DISTANCE COSINE
  WITH TARGET ACCURACY 95
""")
print("workflow_memory created.")

workflow_memory created.


### 2g. `toolbox_memory` (Procedural)

In [9]:
cur.execute("""
CREATE TABLE toolbox_memory (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  tool_name       VARCHAR2(128) NOT NULL,
  tool_schema     JSON          NOT NULL,
  tool_handler    VARCHAR2(256),
  access_policy   JSON,
  usage_notes     CLOB,
  embedding       VECTOR(384, FLOAT32),
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  written_by      VARCHAR2(64)  NOT NULL,
  source_event_id VARCHAR2(64),
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

cur.execute("""
CREATE UNIQUE INDEX idx_toolbox_active
  ON toolbox_memory (
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN tenant_id END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN user_id   END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN agent_id  END,
    CASE WHEN valid_until IS NULL AND deleted_at IS NULL THEN tool_name END
  )
""")
print("toolbox_memory created.")

toolbox_memory created.


### 2h. `knowledge_base_document` + `knowledge_base_chunk` (Semantic)

Note that `knowledge_base_document` stores a `source_uri` **pointer** to the original document (e.g. an object-storage path) plus metadata: title, collection, source type, and provenance. It does **not** store the document's content. The searchable text lives only in `knowledge_base_chunk.content`, chunked and embedded; the original bytes (the PDF, the HTML) stay in object storage, referenced by `source_uri`. The database holds the derived, retrievable form and a pointer home, not the raw file.


In [10]:
cur.execute("""
CREATE TABLE knowledge_base_document (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  collection      VARCHAR2(128) NOT NULL,
  title           VARCHAR2(512) NOT NULL,
  source_uri      VARCHAR2(1024),
  source_type     VARCHAR2(64)  NOT NULL,
  metadata        JSON,
  ingested_at     TIMESTAMP     NOT NULL,
  ingested_by     VARCHAR2(64)  NOT NULL,
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

cur.execute("""
CREATE TABLE knowledge_base_chunk (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  document_id     VARCHAR2(64)  NOT NULL,
  chunk_index     NUMBER(10)    NOT NULL,
  content         CLOB          NOT NULL,
  embedding       VECTOR(384, FLOAT32),
  metadata        JSON,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP,
  CONSTRAINT fk_kb_chunk_doc FOREIGN KEY (document_id) REFERENCES knowledge_base_document (id)
)
""")

cur.execute("""
CREATE UNIQUE INDEX idx_kb_chunk_doc_idx
  ON knowledge_base_chunk (tenant_id, document_id, chunk_index)
""")

cur.execute("""
CREATE INDEX idx_kb_doc_collection
  ON knowledge_base_document (tenant_id, user_id, agent_id, collection, valid_until)
""")

cur.execute("""
CREATE VECTOR INDEX idx_kb_chunk_embedding
  ON knowledge_base_chunk (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH
  DISTANCE COSINE
  WITH TARGET ACCURACY 95
""")
print("knowledge_base_document + knowledge_base_chunk created.")

knowledge_base_document + knowledge_base_chunk created.


### 2i. `deletion_events` (audit)

In [11]:
cur.execute("""
CREATE TABLE deletion_events (
  id          VARCHAR2(64)  DEFAULT SYS_GUID() PRIMARY KEY,
  tenant_id   VARCHAR2(64)  NOT NULL,
  user_id     VARCHAR2(64),
  scope       VARCHAR2(32)  NOT NULL,        -- 'all' | 'tenant' | 'user' | 'thread'
  deleted_at  TIMESTAMP     NOT NULL,
  reason      VARCHAR2(64)  NOT NULL,
  created_at  TIMESTAMP     DEFAULT SYSTIMESTAMP NOT NULL
)
""")
print("deletion_events created.")

# Confirm all tables present.
cur.execute("""
SELECT table_name FROM user_tables
 WHERE LOWER(table_name) IN (
   'guideline_memory','persona_memory','entity_memory','conversation_memory',
   'summarization_memory','workflow_memory','toolbox_memory',
   'knowledge_base_document','knowledge_base_chunk','deletion_events'
 )
 ORDER BY table_name
""")
print("Tables created:", [r[0].lower() for r in cur])
conn.commit()

deletion_events created.
Tables created: ['conversation_memory', 'deletion_events', 'entity_memory', 'guideline_memory', 'knowledge_base_chunk', 'knowledge_base_document', 'persona_memory', 'summarization_memory', 'toolbox_memory', 'workflow_memory']


## 3. Tenant isolation via row-level security

The function returns the predicate `tenant_id = SYS_CONTEXT('memory_ctx','tenant_id')` and the loop attaches it to all nine tenant-scoped data tables (the seven `*_MEMORY` tables plus the two `knowledge_base_*` tables) for SELECT, INSERT, UPDATE, and DELETE. `update_check => TRUE` is what makes the INSERT half real: a careless INSERT with the wrong `tenant_id` is blocked at the statement layer, not just filtered on read. A developer who forgets to filter by tenant can't write a leaking query because the database refuses to run one.

In [12]:
cur.execute("""
CREATE OR REPLACE FUNCTION memory_tenant_policy(
  schema_in IN VARCHAR2, table_in IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
  RETURN q'[
    tenant_id = SYS_CONTEXT('memory_ctx','tenant_id')
  ]';
END;
""")

# Attach the policy to every *_MEMORY table.
cur.execute("""
BEGIN
  FOR rec IN (SELECT table_name FROM user_tables
               WHERE table_name LIKE '%_MEMORY'
                  OR table_name LIKE 'KNOWLEDGE_BASE%')
  LOOP
    BEGIN
      DBMS_RLS.ADD_POLICY(
        object_schema   => USER,
        object_name     => rec.table_name,
        policy_name     => rec.table_name || '_tenant_pol',
        policy_function => 'memory_tenant_policy',
        statement_types => 'SELECT,INSERT,UPDATE,DELETE',
        update_check    => TRUE);
    EXCEPTION WHEN OTHERS THEN
      IF SQLCODE = -28101 THEN NULL; -- policy already exists; idempotent re-run
      ELSE RAISE;
      END IF;
    END;
  END LOOP;
END;
""")

# Confirm policies attached.
cur.execute("""
SELECT object_name, policy_name
  FROM user_policies
 WHERE object_name LIKE '%_MEMORY'
    OR object_name LIKE 'KNOWLEDGE_BASE%'
 ORDER BY object_name
""")
for row in cur:
    print(f"  {row[0]:<25} <- {row[1]}")
conn.commit()

  CONVERSATION_MEMORY       <- CONVERSATION_MEMORY_TENANT_POL
  ENTITY_MEMORY             <- ENTITY_MEMORY_TENANT_POL
  GUIDELINE_MEMORY          <- GUIDELINE_MEMORY_TENANT_POL
  KNOWLEDGE_BASE_CHUNK      <- KNOWLEDGE_BASE_CHUNK_TENANT_POL
  KNOWLEDGE_BASE_DOCUMENT   <- KNOWLEDGE_BASE_DOCUMENT_TENANT_POL
  PERSONA_MEMORY            <- PERSONA_MEMORY_TENANT_POL
  SUMMARIZATION_MEMORY      <- SUMMARIZATION_MEMORY_TENANT_POL
  TOOLBOX_MEMORY            <- TOOLBOX_MEMORY_TENANT_POL
  WORKFLOW_MEMORY           <- WORKFLOW_MEMORY_TENANT_POL


## 4. Set the session context

A real application sets the tenant context once per request from the authenticated identity. Here we set it manually for the rest of the notebook. We also create the `memory_ctx` namespace if it doesn't already exist.

In [13]:
# A trivial package the namespace will be bound to. Oracle requires the
# package to exist before the CREATE CONTEXT statement references it.
cur.execute("""
CREATE OR REPLACE PACKAGE set_memory_ctx AS
  PROCEDURE set_tenant(p_tenant_id IN VARCHAR2);
END;
""")

cur.execute("""
CREATE OR REPLACE PACKAGE BODY set_memory_ctx AS
  PROCEDURE set_tenant(p_tenant_id IN VARCHAR2) IS
  BEGIN
    DBMS_SESSION.SET_CONTEXT('memory_ctx', 'tenant_id', p_tenant_id);
  END;
END;
""")

# Create (or replace) the application context bound to the package above.
cur.execute("""
CREATE OR REPLACE CONTEXT memory_ctx USING set_memory_ctx
""")

TENANT_ACME   = "acme"
TENANT_GLOBEX = "globex"
USER_ID       = "user:jane@acme.com"
AGENT_ID      = "agent:support_v1"

cur.callproc("set_memory_ctx.set_tenant", [TENANT_ACME])
cur.execute("SELECT SYS_CONTEXT('memory_ctx','tenant_id') FROM dual")
print("Active tenant:", cur.fetchone()[0])

Active tenant: acme


## 5. Seed sample data

A small seed: one guideline, one persona, two entities (with real embeddings via `VECTOR_EMBEDDING(ALL_MINILM_L12_V2)`), and two conversation events. Globex's data lives under a separate tenant context so we can demonstrate the boundary later.

In [14]:
def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"

def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

now = datetime.now(timezone.utc).replace(tzinfo=None)

# Guideline: tenant-wide refund threshold for Acme.
gid = new_id("gl")
cur.execute("""
INSERT INTO guideline_memory (
  id, tenant_id, guideline_key, guideline_value,
  version, valid_from, written_by, created_at
) VALUES (
  :id, :tenant_id, 'refund_threshold',
  JSON('{ "max_auto_approve_usd": 500, "min_confidence": 0.6 }'),
  1, :now, 'admin:acme', :now
)
""", id=gid, tenant_id=TENANT_ACME, now=now)

# Persona: Jane prefers terse responses.
pid = new_id("pr")
cur.execute("""
INSERT INTO persona_memory (
  id, tenant_id, user_id, persona_key, persona_value,
  confidence, written_by, valid_from, created_at
) VALUES (
  :id, :tenant_id, :user_id, 'verbosity',
  JSON('{ "value": "terse" }'),
  0.95, 'user', :now, :now
)
""", id=pid, tenant_id=TENANT_ACME, user_id=USER_ID, now=now)

# Entities: two facts with embeddings.
def insert_entity(subject, predicate, content, source_event_id):
    eid = new_id("ent")
    cur.execute("""
    INSERT INTO entity_memory (
      id, tenant_id, user_id,
      subject, predicate, content, content_hash, content_json,
      embedding, confidence, written_by, source_event_id, version,
      valid_from, created_at
    ) VALUES (
      :id, :tenant_id, :user_id,
      :subject, :predicate, :content, :content_hash,
      JSON('{ "entity_type": "product" }'),
      VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :content_for_embed AS DATA),
      0.95, 'agent:support_v1', :source_event_id, 1,
      :now, :now
    )
    """, id=eid, tenant_id=TENANT_ACME, user_id=USER_ID,
         subject=subject, predicate=predicate,
         content=content, content_hash=sha256(content),
         content_for_embed=content,
         source_event_id=source_event_id, now=now)
    return eid

ev1 = new_id("evt")
ev2 = new_id("evt")
ent_webhook = insert_entity(
    "product:stripe", "webhook_signing",
    "Stripe webhooks are signed with HMAC-SHA256 over the raw request body.",
    ev1,
)
ent_rate = insert_entity(
    "product:stripe", "rate_limits",
    "Stripe API rate limits: 100 read req/s and 25 write req/s per account.",
    ev2,
)

# Two conversation events (the source side of the provenance link).
def insert_conv_event(event_id, turn_index, event_type, role, payload):
    cur.execute("""
    INSERT INTO conversation_memory (
      event_id, tenant_id, user_id, thread_id, run_id,
      turn_index, event_type, role, payload, created_at
    ) VALUES (
      :event_id, :tenant_id, :user_id, 'thread_demo', 'run_demo',
      :turn_index, :event_type, :role, JSON(:payload), :now
    )
    """, event_id=event_id, tenant_id=TENANT_ACME, user_id=USER_ID,
         turn_index=turn_index, event_type=event_type, role=role,
         payload=payload, now=now)

insert_conv_event(ev1, 1, "user_msg",  "user",
                  '{ "text": "How are Stripe webhooks signed?" }')
insert_conv_event(ev2, 2, "model_msg", "assistant",
                  '{ "text": "HMAC-SHA256 over the raw request body." }')

# Summarization: one distilled prior-task episode (with an embedding for retrieval).
sum_id = new_id("sum")
sum_text = ("Resolved a Stripe webhook signature mismatch after the customer "
            "rotated their signing secret; redeployed with the new env var.")
cur.execute("""
INSERT INTO summarization_memory (
  id, tenant_id, user_id, task_type, title, summary,
  outcome, embedding, confidence, written_by, source_event_id,
  version, valid_from, completed_at, created_at
) VALUES (
  :id, :tenant_id, :user_id, 'support_case',
  'Stripe webhook signature mismatch after secret rotation',
  :summary, 'resolved',
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :summary AS DATA),
  0.9, 'agent:support_v1', 'run_demo', 1, :now, :now, :now
)
""", id=sum_id, tenant_id=TENANT_ACME, user_id=USER_ID, summary=sum_text, now=now)

# Toolbox: one tenant-wide tool definition (enumerated, like guidelines).
tb_id = new_id("tb")
cur.execute("""
INSERT INTO toolbox_memory (
  id, tenant_id, tool_name, tool_schema, usage_notes,
  version, valid_from, written_by, created_at
) VALUES (
  :id, :tenant_id, 'issue_refund',
  JSON('{ "type": "object", "properties": { "amount_usd": {"type": "number"}, "reason": {"type": "string"} } }'),
  'Issue a customer refund. Requires manager approval above the refund_threshold policy.',
  1, :now, 'admin:acme', :now
)
""", id=tb_id, tenant_id=TENANT_ACME, now=now)

# Workflow: one tenant-wide learned procedure, ranked by description similarity.
wf_id = new_id("wf")
wf_desc = "Diagnose a failing Stripe webhook: check signature, verify the signing secret, confirm the endpoint URL."
cur.execute("""
INSERT INTO workflow_memory (
  id, tenant_id, workflow_name, description, steps,
  success_count, failure_count, embedding, confidence,
  version, valid_from, written_by, created_at
) VALUES (
  :id, :tenant_id, 'diagnose_webhook_failure', :description,
  JSON('["check Stripe-Signature header", "compare dashboard secret to env var", "redeploy", "verify delivery"]'),
  7, 1,
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :description AS DATA),
  0.85, 1, :now, 'agent:support_v1', :now
)
""", id=wf_id, tenant_id=TENANT_ACME, description=wf_desc, now=now)

# Knowledge base: one tenant-wide document plus a single embedded chunk.
kb_doc_id = new_id("kbd")
cur.execute("""
INSERT INTO knowledge_base_document (
  id, tenant_id, collection, title, source_uri, source_type,
  ingested_at, ingested_by, version, valid_from, created_at
) VALUES (
  :id, :tenant_id, 'product-docs',
  'Stripe Webhook Integration Guide',
  's3://acme-kb/product-docs/stripe-webhook-guide.md',   -- pointer, not the bytes
  'markdown',
  :now, 'admin:acme', 1, :now, :now
)
""", id=kb_doc_id, tenant_id=TENANT_ACME, now=now)

kb_chunk_text = ("To verify a Stripe webhook, compute HMAC-SHA256 over the raw request body "
                 "using your signing secret and compare it to the Stripe-Signature header.")
cur.execute("""
INSERT INTO knowledge_base_chunk (
  id, tenant_id, document_id, chunk_index, content, embedding, created_at
) VALUES (
  :id, :tenant_id, :doc_id, 0, :content,
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :content AS DATA), :now
)
""", id=new_id("kbc"), tenant_id=TENANT_ACME, doc_id=kb_doc_id,
     content=kb_chunk_text, now=now)

conn.commit()
print(f"Seeded: 1 guideline, 1 persona, 2 entities, 1 summarization, "
      f"1 toolbox, 1 workflow, 1 KB doc (+1 chunk), "
      f"2 conversation events for tenant={TENANT_ACME!r}.")

Seeded: 1 guideline, 1 persona, 2 entities, 1 summarization, 1 toolbox, 1 workflow, 1 KB doc (+1 chunk), 2 conversation events for tenant='acme'.


### 5a. The tenant boundary is real

Switching the session context to `globex` returns zero rows from Acme's tables. The RLS predicate runs unconditionally on every query.

In [15]:
cur.callproc("set_memory_ctx.set_tenant", [TENANT_GLOBEX])
for tbl in ["guideline_memory", "persona_memory", "entity_memory", "conversation_memory"]:
    cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    print(f"  tenant=globex  {tbl}: {cur.fetchone()[0]} rows")

# Restore Acme context for the rest of the notebook.
cur.callproc("set_memory_ctx.set_tenant", [TENANT_ACME])
print("\nRestored Acme tenant context.")

  tenant=globex  guideline_memory: 0 rows
  tenant=globex  persona_memory: 0 rows
  tenant=globex  entity_memory: 0 rows
  tenant=globex  conversation_memory: 0 rows

Restored Acme tenant context.


## 6. The canonical retrieval predicate

Every LTM read carries the same four-dimension scope filter plus the lifecycle gates. `tenant_id` is enforced by RLS (shown here for clarity), and the other three dimensions inherit upward through `IS NULL` so a tenant-wide guideline matches every user.

In [16]:
cur.execute("""
SELECT id, guideline_key, JSON_SERIALIZE(guideline_value RETURNING VARCHAR2(200))
  FROM guideline_memory
 WHERE tenant_id = SYS_CONTEXT('memory_ctx','tenant_id')  -- enforced by RLS, shown for clarity
   AND (user_id   IS NULL OR user_id   = :user_id)
   AND (agent_id  IS NULL OR agent_id  = :agent_id)
   AND (thread_id IS NULL OR thread_id = :thread_id)
   AND deleted_at IS NULL
   AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
""", user_id=USER_ID, agent_id=AGENT_ID, thread_id="thread_demo")

for row in cur:
    print(f"  {row[1]}: {row[2]}")

  refund_threshold: {"max_auto_approve_usd":500,"min_confidence":0.6}


## 7. Insert pattern for `entity_memory`

A new row enters at `version = 1`, `valid_from = SYSTIMESTAMP`, `source_event_id` pointing back to the conversation event that triggered the write. Provenance is non-negotiable on durable types.

In [17]:
new_event = new_id("evt")
# Source event (simulating a model deciding a new fact)
insert_conv_event(new_event, 3, "model_msg", "assistant",
                  '{ "text": "Acme production DB moved us-east-1 -> eu-west-1." }')

new_entity_id = new_id("ent")
content_v1 = "Production database is in us-east-1, replicas in us-west-2."
cur.execute("""
INSERT INTO entity_memory (
  id, tenant_id, user_id, agent_id, thread_id,
  subject, predicate, content, content_hash, embedding,
  confidence, written_by, source_event_id, version,
  valid_from, created_at
) VALUES (
  :id, :tenant_id, :user_id, NULL, NULL,
  'customer:acme-corp', 'infrastructure',
  :content,
  :content_hash,
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :content AS DATA),
  0.95, 'agent:support_asst', :source_event_id, 1,
  SYSTIMESTAMP, SYSTIMESTAMP
)
""", id=new_entity_id, tenant_id=TENANT_ACME, user_id=USER_ID,
     content=content_v1, content_hash=sha256(content_v1),
     source_event_id=new_event)

conn.commit()
print(f"Inserted entity {new_entity_id} (version 1).")

Inserted entity ent_96422feecdf8 (version 1).


## 8. Supersession (versioned update)

A versioned update inserts a new row at `version = old.version + 1` and stamps the old row's `valid_until` and `superseded_by` in the same transaction. The old row stays queryable for audit; the default retrieval predicate (`valid_until IS NULL`) excludes it.

In [18]:
new_event_v2 = new_id("evt")
insert_conv_event(new_event_v2, 4, "model_msg", "assistant",
                  '{ "text": "Migration confirmed." }')

new_v2_id = new_id("ent")
content_v2 = "Production database is in eu-west-1, replicas in eu-central-1. Migrated 2026-04."

cur.execute("""
DECLARE
  v_now TIMESTAMP := SYSTIMESTAMP;
BEGIN
  INSERT INTO entity_memory (
    id, tenant_id, user_id, agent_id, thread_id,
    subject, predicate, content, content_hash, embedding, confidence,
    written_by, source_event_id, version, valid_from, created_at
  ) VALUES (
    :new_id, :tenant_id, :user_id, NULL, NULL,
    'customer:acme-corp', 'infrastructure',
    :content,
    :content_hash_v2,
    VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :content AS DATA),
    0.97,
    'agent:support_asst', :new_event, 2, v_now, v_now
  );

  UPDATE entity_memory
     SET valid_until = v_now, superseded_by = :new_id
   WHERE id = :old_id AND valid_until IS NULL;
END;
""", new_id=new_v2_id, tenant_id=TENANT_ACME, user_id=USER_ID,
     content=content_v2, content_hash_v2=sha256(content_v2),
     new_event=new_event_v2, old_id=new_entity_id)
conn.commit()

# Verify the supersession chain.
cur.execute("""
SELECT id, version, content, valid_until, superseded_by
  FROM entity_memory
 WHERE subject = 'customer:acme-corp' AND predicate = 'infrastructure'
 ORDER BY version
""")
print("Supersession chain:")
for r in cur:
    iv = r[3].isoformat() if r[3] else "<active>"
    print(f"  v{r[1]} id={r[0]}  valid_until={iv}  superseded_by={r[4]}")

Supersession chain:
  v1 id=ent_96422feecdf8  valid_until=2026-05-29T16:53:39.232652  superseded_by=ent_8474242e5a0c
  v2 id=ent_8474242e5a0c  valid_until=<active>  superseded_by=None


## 9. Expiration via `valid_until`

The default retrieval predicate filters `valid_until IS NULL OR valid_until > SYSTIMESTAMP`, so superseded and expired rows fall out without a DELETE. No row is destroyed; expiration is a query-time concern.

In [19]:
cur.execute("""
SELECT id, version, content
  FROM entity_memory
 WHERE tenant_id = SYS_CONTEXT('memory_ctx','tenant_id')
   AND subject = 'customer:acme-corp' AND predicate = 'infrastructure'
   AND deleted_at IS NULL
   AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
""")

print("Active row(s) under default retrieval predicate:")
for r in cur:
    content_str = r[2].read() if hasattr(r[2], "read") else r[2]
    print(f"  v{r[1]}  id={r[0]}  {content_str[:80]}")

Active row(s) under default retrieval predicate:
  v2  id=ent_8474242e5a0c  Production database is in eu-west-1, replicas in eu-central-1. Migrated 2026-04.


## 10. Unified retrieval: assembling the full context

The agent's Retrieve step pulls every memory type it needs for the prompt in one round trip. Each branch of a `UNION ALL` returns the same shape (`type`, `vec_score`, `lex_score`, `relevance`, `payload`), so guidelines, personas, toolbox definitions, entities, summarizations, workflows, knowledge-base chunks, and the recent conversation tail all come back from a single query plan, with the tenant predicate enforced by RLS on every branch.

Three retrieval styles coexist in one statement, matching how the Retrieve step describes each type:

- **Enumerated** (guideline, persona, toolbox): loaded in full with no score; a policy or tool that applies must apply.
- **Hybrid** (entity, summarization): a vector candidate pool and an Oracle Text `CONTAINS` pool, `FULL OUTER JOIN`ed so each row carries **both** a `vec_score` and a `lex_score`. We show the two signals side by side rather than fusing them into one number; fusion and reranking (`DBMS_VECTOR.RERANK`) are a separate, downstream topic.
- **Vector-only** (workflow, knowledge base): similarity match on the embedding.

The `relevance` column maps the vector score to a tier the application can read directly: `high` ≥ 0.7, `standard` ≥ 0.5, `low` < 0.5. The `type` column tags every row so the application can route each one to the right part of the prompt (static prefix vs. ranked context).

Note the knowledge-base branch returns the matched **chunk** plus its document's `title`/`collection` for citation and a `document_id`/`chunk_index` handle, never the full document body. That keeps the prompt lean and leaves the chunk *addressable*: if the agent decides it needs more, it can fetch the parent document or neighboring chunks on demand rather than paying for them up front.


In [20]:
import re

# Build the lexical query: OR-join the search terms so Oracle Text matches ANY
# of them (CONTAINS treats a bare phrase as a strict sequence otherwise).
def or_terms(text):
    toks = re.findall(r"[A-Za-z0-9]+", text)
    return " OR ".join(toks) if toks else "the"

QUERY = "stripe webhook signature help"
LEX_QUERY = or_terms(QUERY)

# Full context assembly in ONE round trip. Each branch emits the same
# (type, vec_score, lex_score, sort_bucket, payload) shape.
#   - Enumerated types (guideline, persona, toolbox, conversation): no scores.
#   - Hybrid types (entity, summarization): vector AND lexical scores side by side
#     via a FULL OUTER JOIN of a vector candidate pool and an Oracle Text pool.
#     We show both numbers rather than fusing them into one.
#   - Vector-only types (workflow, knowledge base): vector score only.
# The outer query maps the vector score to a relevance tier (Article 2's mapping:
# high >= 0.7, standard >= 0.5, low < 0.5). tenant_id is enforced by RLS on every branch.
cur.execute("""
SELECT type, vec_score, lex_score,
       CASE WHEN vec_score IS NULL     THEN NULL
            WHEN vec_score >= 0.7      THEN 'high'
            WHEN vec_score >= 0.5      THEN 'standard'
            ELSE                            'low'
       END AS relevance,
       payload, sort_bucket
FROM (
  WITH
  guidelines AS (
    SELECT 'guideline' AS type, CAST(NULL AS NUMBER) AS vec_score,
           CAST(NULL AS NUMBER) AS lex_score, 0 AS sort_bucket,
           JSON_OBJECT('guideline_key' VALUE guideline_key,
                       'guideline_value' VALUE guideline_value) AS payload
      FROM guideline_memory
     WHERE deleted_at IS NULL AND valid_until IS NULL
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
       AND (thread_id IS NULL OR thread_id = :thread_id)
  ),
  personas AS (
    SELECT 'persona' AS type, CAST(NULL AS NUMBER) AS vec_score,
           CAST(NULL AS NUMBER) AS lex_score, 1 AS sort_bucket,
           JSON_OBJECT('persona_key' VALUE persona_key,
                       'persona_value' VALUE persona_value,
                       'confidence' VALUE confidence) AS payload
      FROM persona_memory
     WHERE deleted_at IS NULL AND valid_until IS NULL
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
       AND (thread_id IS NULL OR thread_id = :thread_id)
  ),
  toolbox AS (
    SELECT 'toolbox' AS type, CAST(NULL AS NUMBER) AS vec_score,
           CAST(NULL AS NUMBER) AS lex_score, 2 AS sort_bucket,
           JSON_OBJECT('tool_name' VALUE tool_name,
                       'tool_schema' VALUE tool_schema) AS payload
      FROM toolbox_memory
     WHERE deleted_at IS NULL AND valid_until IS NULL
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
  ),
  -- Entity: hybrid. Vector candidate pool + Oracle Text pool, FULL OUTER JOINed.
  entity_vec AS (
    SELECT id, subject, predicate,
           DBMS_LOB.SUBSTR(content, 4000, 1) AS content, confidence,
           VECTOR_DISTANCE(embedding,
             VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE) AS vec_dist
      FROM entity_memory
     WHERE deleted_at IS NULL AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
     ORDER BY vec_dist FETCH FIRST 10 ROWS ONLY
  ),
  entity_lex AS (
    SELECT id, subject, predicate,
           DBMS_LOB.SUBSTR(content, 4000, 1) AS content, confidence,
           SCORE(11) AS lex_raw
      FROM entity_memory
     WHERE deleted_at IS NULL AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
       AND CONTAINS(content, :lex_query, 11) > 0
     ORDER BY lex_raw DESC FETCH FIRST 10 ROWS ONLY
  ),
  entities AS (
    SELECT * FROM (
      SELECT 'entity' AS type,
             CASE WHEN v.vec_dist IS NOT NULL THEN 1.0 / (1.0 + v.vec_dist) END AS vec_score,
             l.lex_raw AS lex_score,   -- raw Oracle Text SCORE (relevance, higher = better)
             3 AS sort_bucket,
             JSON_OBJECT('subject' VALUE COALESCE(v.subject, l.subject),
                         'predicate' VALUE COALESCE(v.predicate, l.predicate),
                         'content' VALUE COALESCE(v.content, l.content),
                         'confidence' VALUE COALESCE(v.confidence, l.confidence)) AS payload
        FROM entity_vec v
        FULL OUTER JOIN entity_lex l ON v.id = l.id
       ORDER BY COALESCE(1.0 / (1.0 + v.vec_dist), 0) DESC
    ) WHERE ROWNUM <= 5
  ),
  -- Summarization: hybrid, same pattern (CONTAINS label 12).
  summ_vec AS (
    SELECT id, title, DBMS_LOB.SUBSTR(summary, 4000, 1) AS summary, outcome,
           VECTOR_DISTANCE(embedding,
             VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE) AS vec_dist
      FROM summarization_memory
     WHERE deleted_at IS NULL AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
     ORDER BY vec_dist FETCH FIRST 10 ROWS ONLY
  ),
  summ_lex AS (
    SELECT id, title, DBMS_LOB.SUBSTR(summary, 4000, 1) AS summary, outcome,
           SCORE(12) AS lex_raw
      FROM summarization_memory
     WHERE deleted_at IS NULL AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
       AND (user_id IS NULL OR user_id = :user_id)
       AND (agent_id IS NULL OR agent_id = :agent_id)
       AND CONTAINS(summary, :lex_query, 12) > 0
     ORDER BY lex_raw DESC FETCH FIRST 10 ROWS ONLY
  ),
  summaries AS (
    SELECT * FROM (
      SELECT 'summarization' AS type,
             CASE WHEN v.vec_dist IS NOT NULL THEN 1.0 / (1.0 + v.vec_dist) END AS vec_score,
             l.lex_raw AS lex_score,   -- raw Oracle Text SCORE
             4 AS sort_bucket,
             JSON_OBJECT('title' VALUE COALESCE(v.title, l.title),
                         'summary' VALUE COALESCE(v.summary, l.summary),
                         'outcome' VALUE COALESCE(v.outcome, l.outcome)) AS payload
        FROM summ_vec v
        FULL OUTER JOIN summ_lex l ON v.id = l.id
       ORDER BY COALESCE(1.0 / (1.0 + v.vec_dist), 0) DESC
    ) WHERE ROWNUM <= 3
  ),
  -- Workflow: similarity (vector only) per the article's Retrieve step.
  workflows AS (
    SELECT * FROM (
      SELECT 'workflow' AS type,
             1.0 / (1.0 + VECTOR_DISTANCE(embedding,
               VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE)) AS vec_score,
             CAST(NULL AS NUMBER) AS lex_score, 5 AS sort_bucket,
             JSON_OBJECT('workflow_name' VALUE workflow_name,
                         'description' VALUE DBMS_LOB.SUBSTR(description, 4000, 1),
                         'success_count' VALUE success_count,
                         'failure_count' VALUE failure_count) AS payload
        FROM workflow_memory
       WHERE deleted_at IS NULL AND (valid_until IS NULL OR valid_until > SYSTIMESTAMP)
         AND (user_id IS NULL OR user_id = :user_id)
         AND (agent_id IS NULL OR agent_id = :agent_id)
       ORDER BY vec_score DESC
    ) WHERE ROWNUM <= 3
  ),
  -- Knowledge base: vector only, over chunks joined to their document.
  knowledge_base AS (
    SELECT * FROM (
      SELECT 'knowledge_base' AS type,
             1.0 / (1.0 + VECTOR_DISTANCE(ch.embedding,
               VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE)) AS vec_score,
             CAST(NULL AS NUMBER) AS lex_score, 6 AS sort_bucket,
             JSON_OBJECT('collection' VALUE d.collection, 'title' VALUE d.title,
                         'document_id' VALUE ch.document_id, 'chunk_index' VALUE ch.chunk_index,
                         'content' VALUE DBMS_LOB.SUBSTR(ch.content, 4000, 1)) AS payload
        FROM knowledge_base_chunk ch
        JOIN knowledge_base_document d ON d.id = ch.document_id
       WHERE ch.deleted_at IS NULL AND d.deleted_at IS NULL
         AND (d.valid_until IS NULL OR d.valid_until > SYSTIMESTAMP)
         AND (d.user_id IS NULL OR d.user_id = :user_id)
       ORDER BY vec_score DESC
    ) WHERE ROWNUM <= 3
  ),
  recent_conversation AS (
    SELECT 'conversation' AS type, CAST(NULL AS NUMBER) AS vec_score,
           CAST(NULL AS NUMBER) AS lex_score, 7 AS sort_bucket,
           JSON_OBJECT('turn_index' VALUE turn_index,
                       'event_type' VALUE event_type,
                       'payload' VALUE payload) AS payload
      FROM (
        SELECT turn_index, event_type, payload
          FROM conversation_memory
         WHERE (user_id IS NULL OR user_id = :user_id)
         ORDER BY turn_index DESC
      ) WHERE ROWNUM <= 5
  )
  SELECT type, vec_score, lex_score, payload, sort_bucket FROM guidelines
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM personas
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM toolbox
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM entities
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM summaries
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM workflows
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM knowledge_base
  UNION ALL SELECT type, vec_score, lex_score, payload, sort_bucket FROM recent_conversation
)
ORDER BY sort_bucket, vec_score DESC NULLS LAST
-- tenant_id predicate appended automatically by RLS on every branch
""", query=QUERY, lex_query=LEX_QUERY, user_id=USER_ID, agent_id=None, thread_id=None)

print(f"lexical query: {LEX_QUERY!r}\n")
print(f"{'type':<16} {'vec':>6} {'lex':>5}  {'relevance':<9}  payload")
print("-" * 104)
for mtype, vec, lex, relevance, payload, _bucket in cur:
    pj = payload.read() if hasattr(payload, "read") else payload
    if isinstance(pj, (dict, list)):
        pj = json.dumps(pj)
    vs = f"{vec:.3f}" if vec is not None else "  -  "
    ls = f"{lex:g}" if lex is not None else "  - "   # raw Oracle Text SCORE
    print(f"{mtype:<16} {vs:>6} {ls:>5}  {(relevance or '—'):<9}  {pj[:52]}")


lexical query: 'stripe OR webhook OR signature OR help'

type                vec   lex  relevance  payload
--------------------------------------------------------------------------------------------------------
guideline           -      -   —          {"guideline_key":"refund_threshold","guideline_value
persona             -      -   —          {"persona_key":"verbosity","persona_value":{"value":
toolbox             -      -   —          {"tool_name":"issue_refund","tool_schema":{"type":"o
entity            0.826     4  high       {"subject":"product:stripe","predicate":"webhook_sig
entity            0.628     4  standard   {"subject":"product:stripe","predicate":"rate_limits
entity            0.531    -   standard   {"subject":"customer:acme-corp","predicate":"infrast
summarization     0.760     3  high       {"title":"Stripe webhook signature mismatch after se
workflow          0.825    -   high       {"workflow_name":"diagnose_webhook_failure","descrip
knowledge_base    0.839    -

### 10a. A policy-governed retrieval example

A narrower look at a single branch: entity retrieval where a guideline supplies the confidence floor. Here `guideline_memory` is joined not as retrieved content but as a *config source*: the query reads `min_confidence` out of the `refund_threshold` policy and applies it as a filter. Vector ranking, a policy-driven threshold, scope inheritance, and tenant isolation, all in one plan.

In [21]:
# The unified retrieval query: vector ranking + scope inheritance + a
# policy-driven confidence floor + tenant isolation, in one statement.
# user_id is supplied as a bind (an app-supplied filter), matching the
# canonical retrieval predicate in Section 6.
cur.execute("""
SELECT  e.id, e.subject, e.content, e.confidence, e.source_event_id,
        VECTOR_DISTANCE(
          e.embedding,
          VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA),
          COSINE
        ) AS vec_dist
  FROM  entity_memory e
  JOIN  guideline_memory g
    ON  g.guideline_key = 'refund_threshold'   -- existing tenant-scoped guideline
   AND  g.valid_until IS NULL
   AND  g.user_id IS NULL
 WHERE  e.deleted_at IS NULL
   AND  (e.valid_until IS NULL OR e.valid_until > SYSTIMESTAMP)
   AND  e.confidence >= JSON_VALUE(g.guideline_value, '$.min_confidence')
   AND  (e.user_id IS NULL OR e.user_id = :user_id)
 ORDER BY vec_dist
 FETCH FIRST 10 ROWS ONLY
-- tenant_id predicate appended automatically by RLS on both tables
""", query="how do stripe webhooks work", user_id=USER_ID)

print("Vector-ranked result (vector + scope + policy + tenant):")
for r in cur:
    content_str = r[2].read() if hasattr(r[2], "read") else r[2]
    print(f"  dist={r[5]:.4f}  conf={r[3]}  {r[1]}/{content_str[:60]}")


Vector-ranked result (vector + scope + policy + tenant):
  dist=0.2474  conf=0.95  product:stripe/Stripe webhooks are signed with HMAC-SHA256 over the raw req
  dist=0.4735  conf=0.95  product:stripe/Stripe API rate limits: 100 read req/s and 25 write req/s pe
  dist=0.9411  conf=0.97  customer:acme-corp/Production database is in eu-west-1, replicas in eu-central-


### 10b. Semantic cache as a vector index over conversation_memory

The semantic cache is a vector index over recent conversation events. The schema is just the index; fusion lives in the retrieval layer. We populate the `payload` column with embeddings of recent events (or you'd add an `embedding` column to `conversation_memory` for this; for the demo we'll skip the actual index creation since `conversation_memory` doesn't have an embedding column in the article's DDL).

This cell is illustrative only. Uncomment and adapt if you extend `conversation_memory` with an embedding column.

In [22]:
# Illustrative DDL (would require adding an embedding column to conversation_memory):
#
# ALTER TABLE conversation_memory ADD (embedding VECTOR(384, FLOAT32));
#
# CREATE VECTOR INDEX idx_conv_semantic
#   ON conversation_memory (embedding)
#   ORGANIZATION INMEMORY NEIGHBOR GRAPH
#   DISTANCE COSINE
#   WITH TARGET ACCURACY 95;
#
# The retrieval-time fusion (vector hits + volatile tail) is a separate concern.
print("See markdown — this cell is illustrative only.")

See markdown — this cell is illustrative only.


## 11. Right-to-forget cascade

One transaction. User-scoped rows get tombstoned. Tenant-scoped derived rows whose `source_event_id` traces back to this user's conversation events also get tombstoned. The conversation events themselves are dropped, and an audit row lands in `deletion_events`. The whole block lands or none of it does.

In [23]:
# First, scope-promote one fact to tenant-wide so we can show the cascade
# also cleans tenant-scoped rows whose lineage points at the user.
ev_tenant = new_id("evt")
insert_conv_event(ev_tenant, 5, "user_msg", "user",
                  '{ "text": "Our SLA is 99.99% uptime." }')
tenant_entity = new_id("ent")
content_tenant = "Acme SLA tier: 99.99% uptime"
cur.execute("""
INSERT INTO entity_memory (
  id, tenant_id, user_id, agent_id, thread_id,
  subject, predicate, content, content_hash, embedding, confidence,
  written_by, source_event_id, version, valid_from, created_at
) VALUES (
  :id, :tenant_id, NULL, NULL, NULL,
  'tenant:acme', 'sla',
  :content, :content_hash,
  VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :content AS DATA),
  0.9, 'agent:support_asst', :ev, 1, SYSTIMESTAMP, SYSTIMESTAMP
)
""", id=tenant_entity, tenant_id=TENANT_ACME,
     content=content_tenant, content_hash=sha256(content_tenant),
     ev=ev_tenant)
conn.commit()

# Now the cascade.
cur.execute("""
DECLARE
  v_now TIMESTAMP := SYSTIMESTAMP;
BEGIN
  -- 1. Hard-delete user-scoped rows in every LTM table.
  UPDATE entity_memory
     SET content = NULL, content_hash = 'erased', embedding = NULL, deleted_at = v_now
   WHERE tenant_id = :tenant_id AND user_id = :user_id;

  UPDATE summarization_memory
     SET summary = NULL, embedding = NULL, deleted_at = v_now
   WHERE tenant_id = :tenant_id AND user_id = :user_id;

  UPDATE persona_memory
     SET persona_value = NULL, deleted_at = v_now
   WHERE tenant_id = :tenant_id AND user_id = :user_id;

  -- 2. Hard-delete tenant-scoped derived rows whose lineage points at the user.
  UPDATE entity_memory e
     SET e.content = NULL, e.content_hash = 'erased', e.embedding = NULL, e.deleted_at = v_now
   WHERE e.tenant_id = :tenant_id
     AND e.user_id IS NULL
     AND EXISTS (
       SELECT 1 FROM conversation_memory c
        WHERE c.event_id = e.source_event_id
          AND c.tenant_id = :tenant_id
          AND c.user_id   = :user_id);

  UPDATE summarization_memory s
     SET s.summary = NULL, s.embedding = NULL, s.deleted_at = v_now
   WHERE s.tenant_id = :tenant_id
     AND s.user_id IS NULL
     AND EXISTS (
       SELECT 1 FROM conversation_memory c
        WHERE c.event_id = s.source_event_id
          AND c.tenant_id = :tenant_id
          AND c.user_id   = :user_id);

  -- 3. Drop the conversation events themselves.
  DELETE FROM conversation_memory
   WHERE tenant_id = :tenant_id AND user_id = :user_id;

  -- 4. Audit-log the erasure.
  INSERT INTO deletion_events (tenant_id, user_id, scope, deleted_at, reason)
  VALUES (:tenant_id, :user_id, 'all', v_now, 'gdpr_erasure');
END;
""", tenant_id=TENANT_ACME, user_id=USER_ID)
conn.commit()

# Confirm what survived.
# The tombstoned tables carry deleted_at; conversation_memory does not — its
# rows are hard-deleted by the cascade (DELETE FROM ...), not tombstoned.
print("After right-to-forget cascade for", USER_ID)
for tbl in ["persona_memory", "entity_memory", "summarization_memory"]:
    cur.execute(f"SELECT COUNT(*) FROM {tbl} WHERE tenant_id = :t AND deleted_at IS NULL",
                t=TENANT_ACME)
    live = cur.fetchone()[0]
    cur.execute(f"SELECT COUNT(*) FROM {tbl} WHERE tenant_id = :t", t=TENANT_ACME)
    total = cur.fetchone()[0]
    print(f"  {tbl}: {live} live / {total} total")

cur.execute("SELECT COUNT(*) FROM conversation_memory WHERE tenant_id = :t", t=TENANT_ACME)
print(f"  conversation_memory: {cur.fetchone()[0]} rows remaining (hard-deleted, not tombstoned)")

cur.execute("SELECT tenant_id, user_id, scope, reason FROM deletion_events ORDER BY created_at DESC FETCH FIRST 1 ROWS ONLY")
print("Audit row:", cur.fetchone())

After right-to-forget cascade for user:jane@acme.com
  persona_memory: 0 live / 1 total
  entity_memory: 0 live / 5 total
  summarization_memory: 0 live / 1 total
  conversation_memory: 0 rows remaining (hard-deleted, not tombstoned)
Audit row: ('acme', 'user:jane@acme.com', 'all', 'gdpr_erasure')


## 12. Cleanup

Drop the policies, drop the tables, close the connection. Skip this cell if you want the seeded tenant to stick around for further exploration.


In [ ]:
# Drop policies (best-effort).
cur.execute("""
BEGIN
  FOR rec IN (SELECT object_name, policy_name FROM user_policies
               WHERE object_name LIKE '%_MEMORY'
                  OR object_name LIKE 'KNOWLEDGE_BASE%')
  LOOP
    BEGIN
      DBMS_RLS.DROP_POLICY(USER, rec.object_name, rec.policy_name);
    EXCEPTION WHEN OTHERS THEN NULL;
    END;
  END LOOP;
END;
""")

# Drop tables.
for t in TABLES:
    try:
        cur.execute(f"DROP TABLE {t} CASCADE CONSTRAINTS PURGE")
    except oracledb.DatabaseError:
        pass

# Drop helper objects.
for stmt in [
    "DROP PACKAGE set_memory_ctx",
    "DROP CONTEXT memory_ctx",
    "DROP FUNCTION memory_tenant_policy",
]:
    try:
        cur.execute(stmt)
    except oracledb.DatabaseError:
        pass

conn.commit()
conn.close()
print("Cleanup complete.")